In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import warnings
import sys
from pathlib import Path
from typing import Dict, List, Any, Optional, Union
import joblib
import json
from datetime import datetime

# Plotting configuration
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
warnings.filterwarnings('ignore')

# Add src to path for imports
sys.path.append(str(Path.cwd().parent))

# Import custom modules
from src.utils.config import load_config, get_model_path, get_results_path
from src.data.make_dataset import load_raw_data
from src.data.preprocessing import preprocess_features

print("✅ Libraries imported successfully!")

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)


In [ ]:
def load_model(model_path: Path):
    """Load a trained model from file."""
    if not model_path.exists():
        raise FileNotFoundError(f"Model file not found: {model_path}")
    
    print(f"📂 Loading model from {model_path}")
    return joblib.load(model_path)

def predict_batch(model, X: pd.DataFrame, include_probabilities: bool = False):
    """Make batch predictions using the trained model."""
    print(f"🔮 Making predictions for {X.shape[0]} samples...")
    
    # Make predictions
    y_pred = model.predict(X)
    
    results = {
        'predictions': y_pred.tolist(),
        'num_predictions': len(y_pred),
        'prediction_shape': y_pred.shape
    }
    
    # Add probabilities if requested and available
    if include_probabilities:
        if hasattr(model, 'predict_proba'):
            y_pred_proba = model.predict_proba(X)
            results['probabilities'] = y_pred_proba.tolist()
            results['probability_shape'] = y_pred_proba.shape
            print("✅ Prediction probabilities included")
        elif hasattr(model, 'decision_function'):
            y_decision = model.decision_function(X)
            results['decision_scores'] = y_decision.tolist()
            results['decision_shape'] = y_decision.shape
            print("✅ Decision function scores included")
        else:
            print("⚠️  Model does not support probability predictions")
    
    return results

def predict_single(model, features: Dict[str, Any], include_probabilities: bool = False):
    """Make a single prediction using the trained model."""
    # Convert features to DataFrame
    X = pd.DataFrame([features])
    
    # Make prediction
    prediction_results = predict_batch(model, X, include_probabilities)
    
    # Extract single prediction
    result = {
        'prediction': prediction_results['predictions'][0],
        'input_features': features
    }
    
    if 'probabilities' in prediction_results:
        result['probabilities'] = prediction_results['probabilities'][0]
    
    if 'decision_scores' in prediction_results:
        result['decision_scores'] = prediction_results['decision_scores'][0]
    
    print(f"🎯 Single prediction: {result['prediction']}")
    
    return result

# Load configuration
try:
    config = load_config('config/config.yaml')
    print("✅ Configuration loaded successfully!")
except FileNotFoundError:
    config = {
        'models': {'best_model': 'xgboost'},
        'data': {'train_file': 'train.csv', 'test_file': 'test.csv'},
        'target_column': 'lengthofstay'
    }
    print("⚠️  Using default configuration")

print(f"🚀 Prediction system initialized at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
